In [21]:
import pandas as pd
import random

macbeth_df = pd.read_fwf('macbeth.txt')
macbeth_df.columns = ['text']

hamlet_df = pd.read_fwf('hamlet.txt')
hamlet_df.columns = ['text']

# Combine the dataframes
df = pd.concat([macbeth_df, hamlet_df], ignore_index=True)
df['text'] = df['text'].astype('str')

In [22]:
# added to remove single letter text which messes up the railfence cipher
df = df[df['text'].str.len() > 1]

In [23]:

for ind in df.index:
    # to lower case
    df['text'][ind] = df['text'][ind].lower()

    # remove whitespace
    df['text'][ind] = df['text'][ind].replace(" ", "")

    # remove punctuation
    df['text'][ind] = df['text'][ind].replace(".", "")
    df['text'][ind] = df['text'][ind].replace(",", "")
    df['text'][ind] = df['text'][ind].replace(";", "")
    df['text'][ind] = df['text'][ind].replace(":", "")
    df['text'][ind] = df['text'][ind].replace("?", "")
    df['text'][ind] = df['text'][ind].replace("!", "")
    df['text'][ind] = df['text'][ind].replace("'", "")
    df['text'][ind] = df['text'][ind].replace('"', "")
    df['text'][ind] = df['text'][ind].replace("(", "")
    df['text'][ind] = df['text'][ind].replace(")", "")
    df['text'][ind] = df['text'][ind].replace("[", "")
    df['text'][ind] = df['text'][ind].replace("]", "")
    df['text'][ind] = df['text'][ind].replace("{", "")
    df['text'][ind] = df['text'][ind].replace("}", "")
    df['text'][ind] = df['text'][ind].replace("-", "")
    df['text'][ind] = df['text'][ind].replace("_", "")
df


,text
0,thetragedyofmacbeth
1,bywilliamshakespeare
2,dramatispersonae
3,duncankingofscotland
4,macbeththaneofglamisandcawdorageneralinthekings
...,...
6811,becomesthefieldbuthereshowsmuchamiss
6812,gobidthesoldiersshoot
6813,exeuntmarchingafterthewhichapealofordnance
6814,areshotoff


# Split 0 -> Caesar Cipher

In [24]:
def caesar_encrypt(text, s=7):
    result = ""
    # transverse the plain text
    for i in range(len(text)):
        char = text[i]
        # Encrypt uppercase characters in plain text
        
        if (char.isupper()):
            result += chr((ord(char) + s-65) % 26 + 65)
        # Encrypt lowercase characters in plain text
        else:
            result += chr((ord(char) + s - 97) % 26 + 97)
    return result

In [25]:
caesar_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
caesar_df['encrypt'] = [caesar_encrypt(text, random.randint(1, 26)) for text in df.text.values] 
caesar_df['algorithm'] = 'caesar'

In [26]:
caesar_df

,algorithm,encrypt
0,caesar,ftqfdmsqpkarymonqft
1,caesar,tqoaddasekzscwkhwsjw
2,caesar,bpykyrgqncpqmlyc
3,caesar,lcvkivsqvownakwbtivl
4,caesar,nbdcfuiuibofpghmbnjtboedbxepsbhfofsbmjouifljoht
...,...,...
6810,caesar,becomesthefieldbuthereshowsmuchamiss
6811,caesar,ltgniymjxtqinjwxxmtty
6812,caesar,wpwmflesjuzafysxlwjlzwozauzshwsdgxgjvfsfuw
6813,caesar,tkxlahmhyy


# Split 1 -> Affine Cipher

## Code for Affine Cipher

In [27]:
class Affine(object):
    def __init__(self):
        pass

    def egcd(self, a, b): 
        x,y, u,v = 0,1, 1,0
        while a != 0: 
            q, r = b//a, b%a 
            m, n = x-u*q, y-v*q 
            b,a, x,y, u,v = a,r, u,v, m,n 
        gcd = b 
        return gcd, x, y 

    def modinv(self, a, m): 
        gcd, x, y = self.egcd(a, m) 
        if gcd != 1: 
            return None # modular inverse does not exist 
        else: 
            return x % m 
    
    def encrypt(self, text, key): 
        #E = (a*x + b) % 26 
        t = text.upper()
        res = []
        for index, c in enumerate(t):
            if c == ' ':
                res.append(' ')
                continue

            enc = ((key[0]*(ord(c) - ord('A')) + key[1]) % 26) + (ord('A') if text[index].isupper() else ord('a'))
            res.append(chr(enc))
        return ''.join(res)


    def decrypt(self, cipher, key): 
        #D(E) = (a^-1 * (E - b)) % 26
        cipher = cipher.upper().replace(' ', '')
        return ''.join([ chr(((self.modinv(key[0], 26)*(ord(c) - ord('A') - key[1])) % 26) + ord('A')) for c in cipher ]) 
      
affine = Affine()

In [28]:
affine_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
affine_df['encrypt'] = [affine.encrypt(text, (random.randint(1, 25), random.randint(1, 25))) for text in df.text.values]
affine_df['algorithm'] = 'affine'

In [29]:
affine_df

,algorithm,encrypt
0,affine,ecieiqeikuogsqmoiec
1,affine,huuuhhuuuuhuuuuhuuhu
2,affine,scbrbwdzipczlobp
3,affine,ecmymmuimwsqqyswamme
4,affine,wmspyrhrhmzycbetmwkomzvsmavclmeyzylmtkzrhyqkzeo
...,...,...
6810,affine,hfphrfnvdfnlfjxhdvdfffndhtnrdpdzrlnn
6811,affine,pjjhbplxtjvbhxxttljjp
6812,affine,lxlrdpbdlhrtdpdnpllprlvrthrdhldzfnfljdddhl
6813,affine,gccomsasoo


# Split 2 -> Vigenere Cipher

In [30]:
MIN_KEY_LEN = 3
MAX_KEY_LEN = 10
ALPHA = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

class Vigenere:
    def __init__(self, key=None):
        if key is None:
            key = self.generate_random_key()
        self.key = key

    def get_key(self):
        return self.key
    
    def set_key(self):
        key = self.generate_random_key()

    def generate_random_key(self) -> str:
        length = random.randint(MIN_KEY_LEN, MAX_KEY_LEN + 1)
        return ''.join(random.sample(ALPHA, length))

    def generate_keyword(self, n: int) -> str:
        return ((n // len(self.key) + 1) * self.key)[:n]

    def get_ord(self, letter):
        return ALPHA.index(letter.upper())
    
    def get_chr(self, num, uppercase=True):
        return ALPHA[num % 26] if uppercase else ALPHA[num % 26].lower()
    
    def encrypt(self, plaintext: str) -> str:
        keyword = self.generate_keyword(len(plaintext))
        return ''.join([self.get_chr(self.get_ord(ch) + self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(plaintext)])

    def decrypt(self, ciphertext: str) -> str:
        keyword = self.generate_keyword(len(ciphertext))
        return ''.join([self.get_chr(self.get_ord(ch) - self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(ciphertext)])

In [31]:
vigenere_test = Vigenere('TESTING')
plaintext = 'this is a test. Hello!'
ciphertext = vigenere_test.encrypt(plaintext)
print(plaintext)
print(ciphertext)

vigenere_test.decrypt(ciphertext)

this is a test. Hello!
mlal vy e mmfz. Zxtyu!


'this is a test. Hello!'

In [32]:
vigenere = Vigenere()

vigenere_df = pd.DataFrame(columns=['algorithm', 'encrypt'])

def vigenere_encrypt_with_random_key(vigenere, text):
    vigenere.set_key()
    return vigenere.encrypt(text)

vigenere_df['encrypt'] = [vigenere_encrypt_with_random_key(vigenere, text) for text in df.text.values]
vigenere_df['algorithm'] = 'vigenere'

vigenere_df

,algorithm,encrypt
0,vigenere,uuyueuhrxzbznnwcrni
1,vigenere,clqjyfjngtuulrmqrusr
2,vigenere,eeunnnjfjfempauf
3,vigenere,ehhdnhlvhhbztpiuyuoq
4,vigenere,nnwcrnigbbaypsamngjfuoqwbjxpeuhrhfeumvhuuylvhhf
...,...,...
6810,vigenere,crwpzytgbfscfyxchnirlffbpjmnhwingjfm
6811,vigenere,hbvjqnirmpyxjrltfbpbn
6812,vigenere,fkyvannnlducotuggysgbfjbjpbbcybyigbleauopy
6813,vigenere,beytuiubzg


# Split 3 -> Substitution Cipher 

In [33]:
def substitution_encrypt(text):
    map = [chr(i+ord('a')) for i in range(26)]
    random.shuffle(map)
    for i in range(26):
        c = chr(i+ord('a'))
        text = text.replace(c, map[i])
    return text

In [34]:
substitution_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
substitution_df['encrypt'] = [substitution_encrypt(text) for text in df.text.values] 
substitution_df['algorithm'] = 'substitution'

In [35]:
substitution_df

,algorithm,encrypt
0,substitution,hchhpdvhvdimjdobhhc
1,substitution,biqhaahlleklifeqfllf
2,substitution,cehfhdhfvfefgohf
3,substitution,dmmmomcemflcimlpaomd
4,substitution,mjprdeueujddaagbjmjmjdbpjjbacjgdddcjbjdeudejdgm
...,...,...
6810,substitution,qdftrdfecdbfdrcqrecdjdfctpfrrfcrrfff
6811,substitution,rjpbcddaejicbameedjjd
6812,substitution,fefndamjnifedijaafnaffkfeifjbfjiaaancdjdif
6813,substitution,llmdecbcdd


# Split 4 -> Railfence Cipher

In [36]:
class RailFence(object):
    def __init__(self):
        pass

    def matrix(self, text, key): 
        try:
            rail = [['\n' for i in range(len(text))] for j in range(key)]
            dir_down = False
            row, col = 0, 0

            for i in range(len(text)):
                if (row == 0) or (row == key - 1):
                    dir_down = not dir_down
                rail[row][col] = text[i]
                col += 1
                if dir_down:
                    row += 1
                else:
                    row -= 1
        except:
            print("ERROR", text, key)

        return rail
    
    def encrypt(self, text, key): 

        rail=self.matrix(text,key)
        result = []
        for i in range(key):
            for j in range(len(text)):
                if rail[i][j] != '\n':
                    result.append(rail[i][j])
        return("" . join(result))

        
rail_fence = RailFence()

In [37]:
rail_fence_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
rail_fence_df['encrypt'] = [rail_fence.encrypt(text, random.randint(2, len(text))) for text in df.text.values]
rail_fence_df['algorithm'] = 'rail_fence'


In [38]:
rail_fence_df

,algorithm,encrypt
0,rail_fence,tmhfaeoctybrdeaetgh
1,rail_fence,bhysaewmkriaealiselp
2,rail_fence,dramatispersonae
3,rail_fence,duncankingofscotdlna
4,rail_fence,macbeththaneofglamisandcawdorageneralintheksign
...,...,...
6810,rail_fence,buemccshowamomehisssteshreefhiteulbd
6811,rail_fence,gsoshbroieodittdhleos
6812,rail_fence,ehcxteneerwauehnntidtfcrmahoagafrnpocielha
6813,rail_fence,aresfhfoot


# Exporting to CSV

In [39]:
def export_df(df, filename):
    df.to_csv(filename, index=False)

# combine all the dataframes
combined_df = pd.concat([caesar_df, affine_df, vigenere_df, rail_fence_df, substitution_df])

# shuffle the data
combined_df = combined_df.sample(frac=1).reset_index(drop=True)

# export the combined dataframe
export_df(combined_df, 'dataset.csv')

# Ciphertext generation

In [40]:
text = "thekingsofscotland"

print("Original Text:", text)
print("Caesar Cipher:", caesar_encrypt(text, random.randint(2, 22)))
print("Affine Cipher:", affine.encrypt(text, (random.randint(2, 24), random.randint(2, 24))))
print("Vigenere Cipher:", vigenere.encrypt(text))
print("Rail Fence Cipher:", rail_fence.encrypt(text, random.randint(3, len(text))))
print("Substitution Cipher:", substitution_encrypt(text))


Original Text: thekingsofscotland
Caesar Cipher: znkqotmyulyiuzrgtj
Affine Cipher: xjztvdxlpllbpxfddn
Vigenere Cipher: uuylvhhfigfwpgfbax
Rail Fence Cipher: thekingsofscotladn
Substitution Cipher: jediualccecncjcqaj
